In [ ]:
!pip install -q torchmetrics torch-fidelity
!pip install -q torchmetrics torch-fidelity lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.4 MB/s eta 0:00:00


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1: V15A — FULL FINAL PAPER RUNS — 3 SEEDS
# ══════════════════════════════════════════════════════════════════════════════
#
# This is the full V15A training cell adapted for multi-seed final paper runs.
#
# It preserves:
#   - static hinge margin = 1.0
#   - N_DIS = 2
#   - mode-seeking regularization
#   - EMA generator
#   - best FID / best Recall checkpoint tracking
#   - per-class metrics
#   - periodic checkpoints
#   - sample generation
#
# Outputs are saved under:
#   /content/drive/MyDrive/lung_gan_checkpoints_v15A_multiseed/seed_XXXX/
#
# ══════════════════════════════════════════════════════════════════════════════

import os
import math
import copy
import random
import zipfile
import time
import gc
import warnings
from collections import deque
import shutil

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils import spectral_norm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm

try:
    from torchmetrics.image.fid import FrechetInceptionDistance
except ImportError:
    !pip install -q torchmetrics
    from torchmetrics.image.fid import FrechetInceptionDistance

try:
    import lpips
except ImportError:
    !pip install -q lpips
    import lpips

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
SEEDS = [42, 123, 777]
EPOCHS = 200

IMG_SIZE = 128
BATCH_SIZE = 64
Z_DIM = 128
EMBED_DIM = 128
BASE_CH = 512
D_BASE_CH = 64
CH_FLOOR = 64

LR_G = 2e-4
LR_D = 2e-4
BETAS = (0.0, 0.9)
EMA_DECAY = 0.999

USE_CLAHE = True
USE_HFLIP = True

# V15A static training settings
BASE_MARGIN = 1.0
N_DIS = 2
USE_MODE_SEEKING = True

# Updated V15A baseline used in your latest run.
# Change to 0.10 if you want the original V15A.
MS_LAMBDA = 0.06

COMPUTE_METRICS = True
METRICS_EVERY = 25
SAMPLE_EVERY = 5
CHECKPOINT_EVERY = 25
METRICS_NUM_SAMPLES = 2000
PR_K = 3
LPIPS_NUM_PAIRS = 300
COMPUTE_PER_CLASS_METRICS = True

SAVE_SAMPLES = True
SAVE_HISTORY = True

CKPT_ROOT = "/content/drive/MyDrive/lung_gan_checkpoints_v15A_multiseed"
os.makedirs(CKPT_ROOT, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ══════════════════════════════════════════════════════════════════════════════
# SEED UTIL
# ══════════════════════════════════════════════════════════════════════════════
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Keep original-style speed behavior.
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# ══════════════════════════════════════════════════════════════════════════════
# MOUNT DRIVE & DATASET
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import drive

mount_path = "/content/drive"
# Check if the mountpoint exists, is a directory, and is not empty
if os.path.exists(mount_path) and os.path.isdir(mount_path) and os.listdir(mount_path):
    print(f"Mountpoint {mount_path} not empty, attempting to clear it.")
    # Use shutil.rmtree to remove non-empty directory
    shutil.rmtree(mount_path)
    # Recreate the directory so drive.mount has a place to mount
    os.makedirs(mount_path)

drive.mount(mount_path, force_remount=True)

ZIP_PATH = "/content/drive/MyDrive/Lung Segmentation Data V2.zip"
EXTRACT_ROOT = "/content/lung_data"

if not os.path.exists(EXTRACT_ROOT):
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_ROOT)

data_root = None
for root, dirs, files in os.walk(EXTRACT_ROOT):
    if "Train" in dirs:
        data_root = os.path.join(root, "Train")
        break

if data_root is None:
    raise FileNotFoundError("Could not find Train folder.")

print("Data root:", data_root)

# ══════════════════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════════════════
class CLAHETransform:
    def __call__(self, img):
        img_np = np.array(img)

        if len(img_np.shape) == 3:
            lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)

            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            l = clahe.apply(l)

            lab = cv2.merge((l, a, b))
            img_np = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        else:
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            img_np = clahe.apply(img_np)

        return Image.fromarray(img_np)


tfm_list = [transforms.Resize((IMG_SIZE, IMG_SIZE))]

if USE_CLAHE:
    tfm_list.append(CLAHETransform())

if USE_HFLIP:
    tfm_list.append(transforms.RandomHorizontalFlip(p=0.5))

tfm_list += [
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
]

transform = transforms.Compose(tfm_list)
dataset = datasets.ImageFolder(data_root, transform=transform)

NUM_CLASSES = len(dataset.classes)
print(f"Classes ({NUM_CLASSES}): {dataset.classes}")
print(f"Dataset size: {len(dataset)}")

# ══════════════════════════════════════════════════════════════════════════════
# MODELS
# ══════════════════════════════════════════════════════════════════════════════
class ConditionalBatchNorm2d(nn.Module):
    def __init__(self, num_features, embed_dim):
        super().__init__()
        self.num_features = num_features

        self.bn = nn.BatchNorm2d(num_features, affine=False)

        self.gamma_fc = nn.Linear(embed_dim, num_features)
        self.beta_fc = nn.Linear(embed_dim, num_features)

        nn.init.normal_(self.gamma_fc.weight, 0.0, 0.02)
        nn.init.ones_(self.gamma_fc.bias)

        nn.init.normal_(self.beta_fc.weight, 0.0, 0.02)
        nn.init.zeros_(self.beta_fc.bias)

    def forward(self, x, class_embed):
        out = self.bn(x)

        gamma = self.gamma_fc(class_embed).view(-1, self.num_features, 1, 1)
        beta = self.beta_fc(class_embed).view(-1, self.num_features, 1, 1)

        return out * gamma + beta


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, embed_dim):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn = ConditionalBatchNorm2d(out_ch, embed_dim)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, class_embed):
        x = self.up(x)
        x = self.conv(x)
        x = self.bn(x, class_embed)
        return self.act(x)


class ConditionalGenerator(nn.Module):
    def __init__(
        self,
        z_dim,
        embed_dim,
        num_classes,
        base_ch,
        img_size,
        ch_floor=CH_FLOOR,
    ):
        super().__init__()

        self.init_size = 4
        self.base_ch = base_ch
        self.embed_dim = embed_dim

        num_stages = int(math.log2(img_size // self.init_size))
        assert self.init_size * (2 ** num_stages) == img_size

        self.label_embed = nn.Embedding(num_classes, embed_dim)
        nn.init.normal_(self.label_embed.weight, 0.0, 0.02)

        self.fc = nn.Linear(
            z_dim + embed_dim,
            base_ch * self.init_size * self.init_size,
        )

        channels = [base_ch]
        ch = base_ch

        for _ in range(num_stages):
            ch = max(ch // 2, ch_floor)
            channels.append(ch)

        self.up_blocks = nn.ModuleList()

        in_ch = base_ch
        for out_ch in channels[1:]:
            self.up_blocks.append(UpBlock(in_ch, out_ch, embed_dim))
            in_ch = out_ch

        self.to_rgb = nn.Sequential(
            nn.Conv2d(in_ch, 3, kernel_size=3, padding=1),
            nn.Tanh(),
        )

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        name = m.__class__.__name__

        if "Conv" in name or "Linear" in name:
            nn.init.normal_(m.weight.data, 0.0, 0.02)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias.data)

        elif "BatchNorm" in name:
            if getattr(m, "weight", None) is not None:
                nn.init.normal_(m.weight.data, 1.0, 0.02)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias.data)

    def forward(self, z, labels):
        class_embed = self.label_embed(labels)

        x = torch.cat([z, class_embed], dim=1)
        x = self.fc(x).view(-1, self.base_ch, self.init_size, self.init_size)

        for block in self.up_blocks:
            x = block(x, class_embed)

        return self.to_rgb(x)


class ProjectionDiscriminator(nn.Module):
    def __init__(self, num_classes, img_size, base_ch):
        super().__init__()

        num_stages = int(math.log2(img_size // 4))

        layer_channels = []
        ch = base_ch

        for _ in range(num_stages):
            layer_channels.append(ch)
            ch = min(ch * 2, 512)

        blocks = []
        in_ch = 3

        for out_ch in layer_channels:
            conv = nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1)
            nn.init.normal_(conv.weight, 0.0, 0.02)

            blocks.append(spectral_norm(conv))
            blocks.append(nn.LeakyReLU(0.2, inplace=True))

            in_ch = out_ch

        self.features = nn.Sequential(*blocks)
        self.feature_dim = in_ch
        self.pool = nn.AdaptiveAvgPool2d(1)

        fc = nn.Linear(self.feature_dim, 1)
        nn.init.normal_(fc.weight, 0.0, 0.02)
        nn.init.zeros_(fc.bias)
        self.fc = spectral_norm(fc)

        embed = nn.Embedding(num_classes, self.feature_dim)
        nn.init.normal_(embed.weight, 0.0, 0.02)
        self.embed = spectral_norm(embed)

    def forward(self, x, labels):
        h = self.features(x)
        h = self.pool(h).flatten(1)

        out = self.fc(h).view(-1)
        proj = (self.embed(labels) * h).sum(dim=1)

        return out + proj


class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()

        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for s_p, p in zip(self.shadow.parameters(), model.parameters()):
            s_p.mul_(self.decay).add_(p.detach(), alpha=1.0 - self.decay)

        for s_b, b in zip(self.shadow.buffers(), model.buffers()):
            s_b.copy_(b)

# ══════════════════════════════════════════════════════════════════════════════
# LOSSES
# ══════════════════════════════════════════════════════════════════════════════
def d_hinge_loss(real_pred, fake_pred, margin=BASE_MARGIN):
    return F.relu(margin - real_pred).mean() + F.relu(margin + fake_pred).mean()


def g_hinge_loss(fake_pred):
    return -fake_pred.mean()


def mode_seeking_loss(img1, img2, z1, z2, eps=1e-5):
    img_dist = torch.mean(torch.abs(img1 - img2))
    z_dist = torch.mean(torch.abs(z1 - z2))

    lz = img_dist / (z_dist + eps)

    return 1.0 / (lz + eps), lz

# ══════════════════════════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def _knn_manifold_precision_recall(real_feats, fake_feats, k=3):
    def knn_radii(feats, k):
        d = torch.cdist(feats, feats)
        d.fill_diagonal_(float("inf"))
        return d.topk(k, largest=False).values[:, -1]

    real_radii = knn_radii(real_feats, k)
    fake_radii = knn_radii(fake_feats, k)

    d_fake_to_real = torch.cdist(fake_feats, real_feats)
    precision = (d_fake_to_real <= real_radii.unsqueeze(0)).any(dim=1).float().mean().item()

    d_real_to_fake = torch.cdist(real_feats, fake_feats)
    recall = (d_real_to_fake <= fake_radii.unsqueeze(0)).any(dim=1).float().mean().item()

    return precision, recall


@torch.no_grad()
def _lpips_diversity(
    generator,
    lpips_metric,
    device,
    num_classes,
    z_dim,
    n_pairs=300,
    batch=32,
    fixed_class=None,
):
    dists = []
    remaining = n_pairs

    while remaining > 0:
        b = min(batch, remaining)

        if fixed_class is None:
            labels = torch.randint(0, num_classes, (b,), device=device)
        else:
            labels = torch.full((b,), fixed_class, device=device, dtype=torch.long)

        z1 = torch.randn(b, z_dim, device=device)
        z2 = torch.randn(b, z_dim, device=device)

        img1 = generator(z1, labels)
        img2 = generator(z2, labels)

        dists.append(lpips_metric(img1, img2).view(-1))

        remaining -= b

    return torch.cat(dists).mean().item()


def compute_metrics(
    generator,
    real_loader,
    device,
    fid_metric,
    lpips_metric,
    num_samples=2000,
    z_dim=128,
    num_classes=2,
    pr_k=3,
    lpips_num_pairs=300,
    fixed_class=None,
):
    generator.eval()

    proc_batch = 16

    fid_metric.reset()

    real_feats = []
    fake_feats = []

    loader_iter = iter(real_loader)

    with torch.no_grad():
        for _ in range(num_samples // proc_batch):
            try:
                real_imgs, real_labels = next(loader_iter)
            except StopIteration:
                break

            real_imgs = real_imgs[:proc_batch].to(device)
            real_labels = real_labels[:proc_batch].to(device)

            z = torch.randn(real_imgs.size(0), z_dim, device=device)
            fake_imgs = generator(z, real_labels)

            real_uint8 = ((real_imgs + 1) / 2 * 255).clamp(0, 255).to(torch.uint8)
            fake_uint8 = ((fake_imgs + 1) / 2 * 255).clamp(0, 255).to(torch.uint8)

            fid_metric.update(real_uint8, real=True)
            fid_metric.update(fake_uint8, real=False)

            real_feats.append(fid_metric.inception(real_uint8).float())
            fake_feats.append(fid_metric.inception(fake_uint8).float())

    fid_score = fid_metric.compute().item()

    real_feats = torch.cat(real_feats, dim=0)
    fake_feats = torch.cat(fake_feats, dim=0)

    precision, recall = _knn_manifold_precision_recall(
        real_feats,
        fake_feats,
        k=pr_k,
    )

    lpips_div = _lpips_diversity(
        generator,
        lpips_metric,
        device,
        num_classes,
        z_dim,
        n_pairs=lpips_num_pairs,
        fixed_class=fixed_class,
    )

    generator.train()

    return fid_score, precision, recall, lpips_div

# ══════════════════════════════════════════════════════════════════════════════
# CHECKPOINT
# ══════════════════════════════════════════════════════════════════════════════
def save_checkpoint(path, epoch_num, G, D, G_ema, opt_G, opt_D, last_metrics):
    torch.save({
        "epoch": epoch_num,
        "G": G.state_dict(),
        "D": D.state_dict(),
        "G_ema": G_ema.shadow.state_dict(),
        "opt_G": opt_G.state_dict(),
        "opt_D": opt_D.state_dict(),
        "classes": dataset.classes,
        "fid": last_metrics["fid"],
        "precision": last_metrics["precision"],
        "recall": last_metrics["recall"],
        "lpips_diversity": last_metrics["lpips_diversity"],
    }, path)

# ══════════════════════════════════════════════════════════════════════════════
# MULTI-SEED TRAINING
# ══════════════════════════════════════════════════════════════════════════════
all_results = []

for seed in SEEDS:
    print("\n" + "=" * 80)
    print(f"V15A | SEED {seed}")
    print("=" * 80)

    set_seed(seed)

    seed_dir = os.path.join(CKPT_ROOT, f"seed_{seed}")
    os.makedirs(seed_dir, exist_ok=True)

    g = torch.Generator()
    g.manual_seed(seed)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        drop_last=True,
        pin_memory=True,
        persistent_workers=True,
        worker_init_fn=seed_worker,
        generator=g,
    )

    class_loaders = {}
    if COMPUTE_PER_CLASS_METRICS:
        targets = np.array(dataset.targets)

        for c in range(NUM_CLASSES):
            idxs = np.where(targets == c)[0].tolist()

            class_loaders[c] = DataLoader(
                Subset(dataset, idxs),
                batch_size=BATCH_SIZE,
                shuffle=True,
                num_workers=2,
                drop_last=True,
                pin_memory=True,
                persistent_workers=True,
                worker_init_fn=seed_worker,
                generator=g,
            )

    G = ConditionalGenerator(
        Z_DIM,
        EMBED_DIM,
        NUM_CLASSES,
        BASE_CH,
        IMG_SIZE,
    ).to(device)

    D = ProjectionDiscriminator(
        NUM_CLASSES,
        IMG_SIZE,
        D_BASE_CH,
    ).to(device)

    G_ema = EMA(G, EMA_DECAY)

    opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=BETAS)
    opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=BETAS)

    fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(device)

    lpips_metric = lpips.LPIPS(net="alex").to(device)
    for p in lpips_metric.parameters():
        p.requires_grad_(False)

    fixed_z = torch.randn(NUM_CLASSES * 4, Z_DIM, device=device)
    fixed_labels = torch.arange(NUM_CLASSES, device=device).repeat_interleave(4)[:NUM_CLASSES * 4]

    metric_history = {
        "epoch": [],
        "fid": [],
        "precision": [],
        "recall": [],
        "lpips_diversity": [],
    }

    metric_history["per_class"] = {
        cname: {
            "epoch": [],
            "fid": [],
            "precision": [],
            "recall": [],
            "lpips_diversity": [],
        }
        for cname in dataset.classes
    }

    per_class_recall_leader_history = []

    loss_history = {
        "epoch": [],
        "D": [],
        "G": [],
        "MS_ratio": [],
        "D_real": [],
        "D_fake": [],
    }

    best_recall = -1.0
    best_fid = float("inf")

    best_recall_info = {}
    best_fid_info = {}

    last_metrics = {
        "fid": None,
        "precision": None,
        "recall": None,
        "lpips_diversity": None,
    }

    start_time = time.time()

    for epoch in range(EPOCHS):
        d_loss_sum = 0.0
        g_loss_sum = 0.0
        ms_lz_sum = 0.0
        real_pred_sum = 0.0
        fake_pred_sum = 0.0
        n_batches = 0

        pbar = tqdm(loader, desc=f"Seed {seed} Epoch {epoch + 1}/{EPOCHS}")

        for real_imgs, labels in pbar:
            real_imgs = real_imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            bs = real_imgs.size(0)

            # Discriminator
            for _ in range(N_DIS):
                z = torch.randn(bs, Z_DIM, device=device)

                with torch.no_grad():
                    fake_imgs = G(z, labels)

                d_real = D(real_imgs, labels)
                d_fake = D(fake_imgs, labels)

                d_loss = d_hinge_loss(d_real, d_fake, margin=BASE_MARGIN)

                opt_D.zero_grad(set_to_none=True)
                d_loss.backward()
                nn.utils.clip_grad_norm_(D.parameters(), max_norm=1000)
                opt_D.step()

            # Generator
            z = torch.randn(bs, Z_DIM, device=device)
            fake_imgs = G(z, labels)

            g_loss = g_hinge_loss(D(fake_imgs, labels))

            ms_lz_val = torch.tensor(0.0, device=device)

            if USE_MODE_SEEKING:
                z2 = torch.randn(bs, Z_DIM, device=device)
                fake_imgs2 = G(z2, labels)

                ms_term, ms_lz_val = mode_seeking_loss(fake_imgs, fake_imgs2, z, z2)
                g_loss = g_loss + MS_LAMBDA * ms_term

            opt_G.zero_grad(set_to_none=True)
            g_loss.backward()
            nn.utils.clip_grad_norm_(G.parameters(), max_norm=1000)
            opt_G.step()

            G_ema.update(G)

            d_loss_sum += d_loss.item()
            g_loss_sum += g_loss.item()
            ms_lz_sum += ms_lz_val.item()

            real_pred_sum += d_real.mean().item()
            fake_pred_sum += d_fake.mean().item()

            n_batches += 1

            pbar.set_postfix(
                D_loss=d_loss.item(),
                G_loss=g_loss.item(),
                MS_ratio=ms_lz_val.item(),
            )

        avg_d = d_loss_sum / n_batches
        avg_g = g_loss_sum / n_batches
        avg_ms = ms_lz_sum / n_batches
        avg_real = real_pred_sum / n_batches
        avg_fake = fake_pred_sum / n_batches

        loss_history["epoch"].append(epoch + 1)
        loss_history["D"].append(avg_d)
        loss_history["G"].append(avg_g)
        loss_history["MS_ratio"].append(avg_ms)
        loss_history["D_real"].append(avg_real)
        loss_history["D_fake"].append(avg_fake)

        print(
            f"Seed {seed} Epoch {epoch + 1}/{EPOCHS} | "
            f"D={avg_d:.4f} | G={avg_g:.4f} | "
            f"D(real)={avg_real:.3f} D(fake)={avg_fake:.3f} | "
            f"MS_ratio={avg_ms:.4f}"
        )

        # Metrics
        if COMPUTE_METRICS and ((epoch + 1) % METRICS_EVERY == 0 or epoch == 0):
            fid_val, prec_val, rec_val, lpips_val = compute_metrics(
                G_ema.shadow,
                loader,
                device,
                fid_metric,
                lpips_metric,
                num_samples=METRICS_NUM_SAMPLES,
                z_dim=Z_DIM,
                num_classes=NUM_CLASSES,
                pr_k=PR_K,
                lpips_num_pairs=LPIPS_NUM_PAIRS,
            )

            metric_history["epoch"].append(epoch + 1)
            metric_history["fid"].append(fid_val)
            metric_history["precision"].append(prec_val)
            metric_history["recall"].append(rec_val)
            metric_history["lpips_diversity"].append(lpips_val)

            print(
                f"  Metrics: FID={fid_val:.2f}  P={prec_val:.4f}  "
                f"R={rec_val:.4f}  LPIPS={lpips_val:.4f}"
            )

            last_metrics.update(
                fid=fid_val,
                precision=prec_val,
                recall=rec_val,
                lpips_diversity=lpips_val,
            )

            if rec_val > best_recall:
                best_recall = rec_val

                best_recall_info = {
                    "epoch": epoch + 1,
                    "fid": fid_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "lpips_diversity": lpips_val,
                }

                save_checkpoint(
                    os.path.join(seed_dir, "lung_gan_best_recall.pth"),
                    epoch + 1,
                    G,
                    D,
                    G_ema,
                    opt_G,
                    opt_D,
                    last_metrics,
                )

                print(f"  -> new best Recall ({rec_val:.4f})")

            if fid_val < best_fid:
                best_fid = fid_val

                best_fid_info = {
                    "epoch": epoch + 1,
                    "fid": fid_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "lpips_diversity": lpips_val,
                }

                save_checkpoint(
                    os.path.join(seed_dir, "lung_gan_best_fid.pth"),
                    epoch + 1,
                    G,
                    D,
                    G_ema,
                    opt_G,
                    opt_D,
                    last_metrics,
                )

                print(f"  -> new best FID ({fid_val:.2f})")

            if COMPUTE_PER_CLASS_METRICS:
                for c, cname in enumerate(dataset.classes):
                    n_avail = len(class_loaders[c].dataset)

                    f_c, p_c, r_c, l_c = compute_metrics(
                        G_ema.shadow,
                        class_loaders[c],
                        device,
                        fid_metric,
                        lpips_metric,
                        num_samples=min(METRICS_NUM_SAMPLES, n_avail),
                        z_dim=Z_DIM,
                        num_classes=NUM_CLASSES,
                        pr_k=PR_K,
                        lpips_num_pairs=LPIPS_NUM_PAIRS,
                        fixed_class=c,
                    )

                    metric_history["per_class"][cname]["epoch"].append(epoch + 1)
                    metric_history["per_class"][cname]["fid"].append(f_c)
                    metric_history["per_class"][cname]["precision"].append(p_c)
                    metric_history["per_class"][cname]["recall"].append(r_c)
                    metric_history["per_class"][cname]["lpips_diversity"].append(l_c)

                    print(
                        f"    [{cname:10s}] FID={f_c:.2f}  P={p_c:.4f}  "
                        f"R={r_c:.4f}  LPIPS={l_c:.4f}"
                    )

                recalls = {
                    cn: metric_history["per_class"][cn]["recall"][-1]
                    for cn in dataset.classes
                }

                leader = max(recalls, key=recalls.get)
                gap_leader = max(recalls.values()) - min(recalls.values())

                print(f"    Recall leader: {leader} (+{gap_leader:.4f})")

                per_class_recall_leader_history.append(
                    (epoch + 1, leader, gap_leader)
                )

        # Samples
        if SAVE_SAMPLES and ((epoch + 1) % SAMPLE_EVERY == 0 or epoch == 0):
            G_ema.shadow.eval()

            with torch.no_grad():
                fake = G_ema.shadow(fixed_z, fixed_labels).detach().cpu()

            grid = make_grid(fake, nrow=4, normalize=True, value_range=(-1, 1))

            plt.figure(figsize=(8, 8))
            plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
            plt.title(f"V15A Seed {seed} EMA samples — epoch {epoch + 1}")
            plt.axis("off")

            plt.savefig(
                os.path.join(seed_dir, f"samples_epoch_{epoch + 1}.png"),
                bbox_inches="tight",
            )
            plt.close()

            G_ema.shadow.train()

        # Periodic checkpoint
        if (epoch + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(
                os.path.join(seed_dir, f"lung_gan_epoch_{epoch + 1}.pth"),
                epoch + 1,
                G,
                D,
                G_ema,
                opt_G,
                opt_D,
                last_metrics,
            )

    # Final checkpoint
    save_checkpoint(
        os.path.join(seed_dir, "lung_gan_final.pth"),
        EPOCHS,
        G,
        D,
        G_ema,
        opt_G,
        opt_D,
        last_metrics,
    )

    if SAVE_HISTORY:
        torch.save(
            {
                "metric_history": metric_history,
                "loss_history": loss_history,
                "per_class_recall_leader_history": per_class_recall_leader_history,
                "best_recall_info": best_recall_info,
                "best_fid_info": best_fid_info,
            },
            os.path.join(seed_dir, f"history_seed_{seed}.pt"),
        )

    total_time_min = (time.time() - start_time) / 60.0

    all_results.append({
        "seed": seed,
        "best_fid": best_fid,
        "best_recall": best_recall,
        "best_fid_precision": best_fid_info.get("precision", np.nan),
        "best_fid_recall": best_fid_info.get("recall", np.nan),
        "best_fid_lpips": best_fid_info.get("lpips_diversity", np.nan),
        "best_recall_fid": best_recall_info.get("fid", np.nan),
        "best_recall_precision": best_recall_info.get("precision", np.nan),
        "best_recall_lpips": best_recall_info.get("lpips_diversity", np.nan),
        "time_min": total_time_min,
    })

    print(f"Seed {seed} complete. Time: {total_time_min:.1f} min")

    del G, D, G_ema, opt_G, opt_D, fid_metric, lpips_metric
    del loader, class_loaders

    gc.collect()
    torch.cuda.empty_cache()

# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
df = pd.DataFrame(all_results)

print("\n" + "=" * 80)
print("V15A FINAL MULTI-SEED SUMMARY")
print("=" * 80)
print(df.to_string(index=False))

print("\nMean ± Std:")
print(f"Best FID:             {df['best_fid'].mean():.2f} ± {df['best_fid'].std():.2f}")
print(f"Best Recall:          {df['best_recall'].mean():.4f} ± {df['best_recall'].std():.4f}")
print(f"Best-FID Precision:   {df['best_fid_precision'].mean():.4f} ± {df['best_fid_precision'].std():.4f}")
print(f"Best-FID LPIPS:       {df['best_fid_lpips'].mean():.4f} ± {df['best_fid_lpips'].std():.4f}")
print(f"Best-Recall FID:      {df['best_recall_fid'].mean():.2f} ± {df['best_recall_fid'].std():.2f}")
print(f"Best-Recall Precision:{df['best_recall_precision'].mean():.4f} ± {df['best_recall_precision'].std():.4f}")
print(f"Best-Recall LPIPS:    {df['best_recall_lpips'].mean():.4f} ± {df['best_recall_lpips'].std():.4f}")
print(f"Time (min):           {df['time_min'].mean():.1f} ± {df['time_min'].std():.1f}")

df.to_csv(os.path.join(CKPT_ROOT, "v15A_multiseed_results.csv"), index=False)
print(f"\nSaved summary CSV to: {os.path.join(CKPT_ROOT, 'v15A_multiseed_results.csv')}")


Device: cuda
Mountpoint /content/drive not empty, attempting to clear it.
Mounted at /content/drive
Data root: /content/lung_data/Lung Segmentation Data V2/Train
Classes (2): ['COVID-19', 'Normal']
Dataset size: 14507

V15A | SEED 42


Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 222MB/s]


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 211MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


...

Seed 777 Epoch 198/200 | D=1.6873 | G=0.4105 | D(real)=0.165 D(fake)=-0.176 | MS_ratio=0.4060


Seed 777 Epoch 199/200:   0%|          | 0/226 [00:00<?, ?it/s]

Seed 777 Epoch 199/200 | D=1.6848 | G=0.4843 | D(real)=0.238 D(fake)=-0.105 | MS_ratio=0.4063


Seed 777 Epoch 200/200:   0%|          | 0/226 [00:00<?, ?it/s]

Seed 777 Epoch 200/200 | D=1.6842 | G=0.4152 | D(real)=0.185 D(fake)=-0.155 | MS_ratio=0.4058
  Metrics: FID=39.62  P=0.6555  R=0.2540  LPIPS=0.3374
    [COVID-19  ] FID=46.81  P=0.6513  R=0.2027  LPIPS=0.3712
    [Normal    ] FID=35.25  P=0.6291  R=0.2477  LPIPS=0.3166
    Recall leader: Normal (+0.0449)
Seed 777 complete. Time: 258.7 min

V15A FINAL MULTI-SEED SUMMARY
 seed  best_fid  best_recall  best_fid_precision  best_fid_recall  best_fid_lpips  best_recall_fid  best_recall_precision  best_recall_lpips   time_min
   42 34.618000       0.2705              0.6670           0.2140        0.343824        38.898613                  0.656           0.342929 259.962461
  123 42.455135       0.2820              0.5710           0.2245        0.330722        44.373856                  0.560           0.344020 258.721721
  777 33.630337       0.2785              0.6595           0.2635        0.335150        42.078499                  0.632           0.334046 258.668372

Mean ± Std:
Best F